<img src="https://s3-us-west-2.amazonaws.com/public.notion-static.com/7fa59d58-ba42-4de1-840a-e2e31ab9ce3b/ba416d9e-ee4d-4b7a-a9fe-a506333af2b7.png" alt="Abstract Banner" width="100%" height="200" style="object-fit: cover; border-radius: 8px;">
by: Elmar Leonard, Muhammad Rafi Andrianto and Valencia

# **Section 0: Project Initialization**

## **0.1 Importing Library**

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## **0.2 Global Configuration**

In [ ]:
DATA_DIR = 'Dataset' 

orders       = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_orders_dataset.csv')
customers    = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_customers_dataset.csv')
order_items  = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_order_items_dataset.csv')
payments     = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_order_payments_dataset.csv')
reviews      = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_order_reviews_dataset.csv')
products     = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_products_dataset.csv')
sellers      = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_sellers_dataset.csv')
geolocation  = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_geolocation_dataset.csv')
cat_translation = pd.read_csv(f'../{DATA_DIR}/Raw Data/product_category_name_translation.csv')

# **Section 1: Data Explanation**

**Dataset Overview**

* **Source File:** `Brazilian E-Commerce Public Dataset by Olist`
* **Total Dataset:** `9 csv`
* **Feature Dimensionality:** `52 columns across the csv`

## **1.1 General Information**

In [3]:
datasets = {
    'orders': orders,
    'customers': customers,
    'items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'category_translation': cat_translation
}

for name, df in datasets.items():
    print("=" * 50)
    print(f" TABLE: {name.upper()} {df.shape}")
    print("=" * 50)
    
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    null_summary = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': missing_pct})
    print(null_summary)
    
    if null_summary['Missing Values'].sum() == 0:
        print("No missing values found!")
        
    print("\nColumn Data Types:")
    print(df.dtypes)
    print("\n")

 TABLE: ORDERS (99441, 8)
                               Missing Values  Percentage (%)
order_id                                    0        0.000000
customer_id                                 0        0.000000
order_status                                0        0.000000
order_purchase_timestamp                    0        0.000000
order_approved_at                         160        0.160899
order_delivered_carrier_date             1783        1.793023
order_delivered_customer_date            2965        2.981668
order_estimated_delivery_date               0        0.000000

Column Data Types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


 TABLE: CUSTOMERS (99441, 5)
                          Missing Values  Percentage (%

In [4]:
total_transactions = customers['customer_id'].nunique()
unique_customers = customers['customer_unique_id'].nunique()

print(f"total transactions (customer_id): {total_transactions}")
print(f"total unique customers (customer_unique_id): {unique_customers}")
print(f"repeat customers count: {total_transactions - unique_customers}")
print(f"repeat customer ratio: {((total_transactions - unique_customers) / total_transactions) * 100:.2f}%")

total transactions (customer_id): 99441
total unique customers (customer_unique_id): 96096
repeat customers count: 3345
repeat customer ratio: 3.36%


In [5]:
print("multi order sample")
multi_item_id = order_items.groupby('order_id').filter(lambda x: len(x) > 1)['order_id'].iloc[3]
display(order_items[order_items['order_id'] == multi_item_id][['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']])

print("multi payment sample")
multi_pay_id = payments.groupby('order_id').filter(lambda x: len(x) > 1)['order_id'].iloc[0]
display(payments[payments['order_id'] == multi_pay_id][['order_id', 'payment_sequential', 'payment_type', 'payment_value']])

multi order sample


,order_id,order_item_id,product_id,price,freight_value
32,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,21.33,15.1
33,00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,21.33,15.1
34,00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,21.33,15.1


multi payment sample


,order_id,payment_sequential,payment_type,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,45.17
57742,5cfd514482e22bc992e7693f0e3e8df7,1,credit_card,665.41


## **1.2 Data Information**

### **1.2.1 olist_orders_dat**


| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order (an order can have multiple items) |
| customer_id | string | Key to the customer who placed the order. |
| order_status | string | Order lifecycle status (delivered, shipped, canceled, etc). |
| order_purchase_timestamp | datetime (string) | Timestamp when the order was placed. |
| order_approved_at | datetime (string) | Timestamp when payment was approved. |
| order_delivered_carrier_date | datetime (string) | Timestamp when the order was handed to the logistics carrier. |
| order_delivered_customer_date | datetime (string) | Timestamp when the order was actually delivered to the customer. |
| order_estimated_delivery_date | datetime (string) | Delivery date estimate given to the customer at purchase time. |

### **1.2.2 olist_customers_dat**

| Column | Type | Description |
| --- | --- | --- |
| customer_id | string | key to the orders dataset. Each order has a unique customer_id. |
| customer_unique_id | string | The actual unique person (use this to track customers across multiple orders) |
| customer_zip_code_prefix | integer | First 5 digits of customer's zip code. |
| customer_city | string | Customer's city. |
| customer_state | string | Customer's state (Brazilian state code, e.g. SP, RJ). |

### **1.2.3 olist_order_items_dat**

| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order. |
| order_item_id | integer | Sequential number identifying number of items within the order (1, 2, 3...). |
| product_id | string | Product unique identifier. |
| seller_id | string | Seller unique identifier. |
| shipping_limit_date | datetime (string) | Seller's deadline to hand the item to the logistic partner (carrier). |
| price | float | Item price. |
| freight_value | float | Freight/shipping cost for this item. (if an order has more than one item the freight value is splitted between items) |

### **1.2.4 olist_order_payments_dat**

| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order. |
| payment_sequential | integer | Sequence number if a customer used more than one payment method for the same order. |
| payment_type | string | Payment method (credit_card, boleto, voucher, debit_card). |
| payment_installments | integer | Number of installments chosen. |
| payment_value | float | Amount paid via this payment row. |

### **1.2.5 olist_order_reviews_dat**

| Column | Type | Description |
| --- | --- | --- |
| review_id | string | Unique identifier of the review. |
| order_id | string | Unique identifier of the order. |
| review_score | integer | Customer's satisfaction rating, 1 (worst) to 5 (best). |
| review_comment_title | string | Title of the written review left by the customer (in portuguese). |
| review_comment_message | string | Comment message left by the customer (in portuguese). |
| review_creation_date | datetime (string) | Date the satisfaction survey was sent to the customer. |
| review_answer_timestamp | datetime (string) | Timestamp when the customer submitted the review. |

### **1.2.6 olist_products_dat**

| Column | Type | Description |
| --- | --- | --- |
| product_id | string | Unique identifier of the product. |
| product_category_name | string | Product category name  (in portuguese). |
| product_name_lenght | float | Number of characters in the product name. |
| product_description_lenght | float | Number of characters in the product description. |
| product_photos_qty | float | Number of photos published for the product. |
| product_weight_g | float | Product weight in grams. |
| product_length_cm | float | Product length in cm. |
| product_height_cm | float | Product height in cm. |
| product_width_cm | float | Product width in cm. |

### **1.2.7 olist_sellers_dat**

| Column | Type | Description |
| --- | --- | --- |
| seller_id | string | Unique identifier of the seller. |
| seller_zip_code_prefix | integer | First 5 digits of seller's zip code. |
| seller_city | string | Seller's city. |
| seller_state | string | Seller's state. |

### **1.2.8 product_category_name_translation**

| Column | Type | Description |
| --- | --- | --- |
| product_category_name | string | Category name in Portuguese. |
| product_category_name_english | string | Same category translated to English. |

### **1.2.9 olist_geolocation_dataset**

| Column | Type | Description |
| --- | --- | --- |
| geolocation_zip_code_prefix | integer | First 5 digits of the zip code. |
| geolocation_lat | float | Latitude of the zip code location. |
| geolocation_lng | float | Longitude of the zip code location. |
| geolocation_city | string | City name for the zip code. |
| geolocation_state | string | State abbreviation for the zip code. |

# **Section 2: Data Merging**

## **2.1 First Merge (orders + order_items)**

Joining `olist_orders_dataset` with `olist_order_items_dataset` expands our dataset from 99,441 rows to 113,425 rows, allowing us to analyze individual product prices and freight charges.

In [6]:
# order_items
df = orders.merge(order_items, on='order_id', how='left')
print('+ order_items: ', df.shape)

display(df.head())

+ order_items:  (113425, 14)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


## **2.2 Product Details & Translation (df + products & cat_translation)**

Joining with `olist_products_dataset` attaches product physical dimensions (weight, length, height, width) and translates Portuguese product category names into English from `product_category_name_translation`.

In [7]:
# products ( & translate category to English)
translate = products.merge(cat_translation, on='product_category_name', how='left')

df = df.merge(translate, on='product_id', how='left')
print('+ products: ', df.shape)
display(df.head())


+ products:  (113425, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


## **2.3 Seller Metadata (df + sellers)**

Adds seller geographic attributes (seller_city, seller_state, seller_zip_code_prefix).

In [8]:
# sellers
df = df.merge(sellers, on='seller_id', how='left')
print('+ sellers: ', df.shape)
display(df.head())

+ sellers:  (113425, 26)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP


## **2.4 Customer Data (df + customers)**

Maps the transaction token (customer_id) to the persistent buyer ID (customer_unique_id) and attaches buyer location details (customer_city, customer_state).

In [9]:
# customers
df = df.merge(customers, on='customer_id', how='left')
print('+ customers: ', df.shape)
display(df.head())

+ customers:  (113425, 30)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


## **2.5 Payments Merge (df + payments_agg)**

Orders can support multiple payment methods, such as splitting costs between credit cards and vouchers. To verify if an order uses multiple primary payment types or just a single voucher, the following test will be conducted:

In [10]:
type_sets = payments.groupby('order_id')['payment_type'].apply(lambda s: set(s))
non_voucher_types = type_sets.apply(lambda s: s - {'voucher'})
non_voucher_count = non_voucher_types.apply(len)

print(non_voucher_count.value_counts())

outliers = non_voucher_count[non_voucher_count >= 2]
print(f'\norders breaking the pattern: {len(outliers)}')
if len(outliers) > 0:
    print(type_sets[outliers.index])

payment_type
1    97818
0     1621
2        1
Name: count, dtype: int64

orders breaking the pattern: 1
order_id
a079628ac8002126e75f86b0f87332e4    {credit_card, debit_card}
Name: payment_type, dtype: object


The test results indicate that `99.99%` of orders used only one major payment type alongside a voucher, or relied entirely on a voucher. Consequently, we can conclude that the main program only supports a single primary payment system. This conclusion is backed by the functionality of the Olist website, which restricts users to selecting one main payment system, with the option to supplement or fulfill the payment using vouchers.  

Based on these findings, we can now proceed to merge the payments dataset. In this step, we will extract the primary payment system, aggregate the total number of vouchers used per order, and calculate the total value contributed by those vouchers.

In [11]:
def _agg_payments(g):
    g = g.sort_values('payment_sequential')
    non_voucher = g[g['payment_type'] != 'voucher']
    voucher_rows = g[g['payment_type'] == 'voucher']

    if len(non_voucher) > 0:
        dominant_type = non_voucher.groupby('payment_type')['payment_value'].sum().idxmax()
        dominant_installments = non_voucher.loc[non_voucher['payment_type'] == dominant_type, 'payment_installments'].max()
    else:
        dominant_type = 'voucher'
        dominant_installments = g['payment_installments'].max()

    return pd.Series({
        'total_payment_value': g['payment_value'].sum(),
        'payment_installments': dominant_installments,
        'payment_type': dominant_type,
        'n_vouchers': len(voucher_rows),
        'voucher_value': voucher_rows['payment_value'].sum()
    })

payments_agg = payments.groupby('order_id').apply(_agg_payments).reset_index()

df = df.merge(payments_agg, on='order_id', how='left')
print('+ payments (agg): ', df.shape)
display(df.head())

+ payments (agg):  (113425, 35)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,credit_card,2.0,20.59
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,0.0,0.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,0.0,0.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,credit_card,0.0,0.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,credit_card,0.0,0.00


## **2.6 Deduplicated Reviews (df + reviews_dedup)**

`551 duplicate` rows across `547 orders (0.56%)` in olist_order_reviews_dataset contain multiple review submissions. Merging reviews directly without handling these duplicate order IDs would multiply those item rows in the final dataset. Sorting chronologically by review_answer_timestamp and keeping the latest record will result in a strict 1-to-1 join per order while retaining the customer's final updated rating.

In [12]:
# reviews: order_id should be ~unique, but merge safely
reviews_dedup = reviews.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')

df = df.merge(reviews_dedup[['order_id', 'review_score', 'review_comment_message',
                              'review_creation_date', 'review_answer_timestamp']],
              on='order_id', how='left')
              
print('+ reviews: ', df.shape)
display(df.head())

+ reviews:  (113425, 39)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,credit_card,2.0,20.59,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,0.0,0.00,4.0,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,0.0,0.00,5.0,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,credit_card,0.0,0.00,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,credit_card,0.0,0.00,5.0,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51


## **2.6 Seller & Customer Geolocation (df + geo_seller + geo_customer)**

The geolocation dataset consists of `1,001,163 rows` containing multiple coordinate entries for each unique zip code. To establish a single, definitive latitude and longitude for each zip code, the data must undergo a cleaning process. First, the dataset will be filtered to retain only the coordinates that fall within Brazil's geographic boundaries. Afterward, the remaining latitude and longitude values will be aggregated to calculate the centroid for each unique zip code.

In [13]:
# geolocation
geolocation = geolocation[(geolocation['geolocation_lat'] >= -33.750833) & (geolocation['geolocation_lat'] <= 5.269444) &
                          (geolocation['geolocation_lng'] >= -73.983056) & (geolocation['geolocation_lng'] <= -34.793056)]

def _mode_or_first(s):
    m = s.mode()
    return m.iat[0] if not m.empty else s.iloc[0]

geolocation = geolocation.groupby('geolocation_zip_code_prefix', as_index=False).agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', _mode_or_first),
    geolocation_state=('geolocation_state', _mode_or_first),
)

df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].astype('Int64')
df['customer_zip_code_prefix'] = df['customer_zip_code_prefix'].astype('Int64')
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype('Int64')

The `geo_seller` dataset was created by renaming geolocation columns to prevent naming conflicts with the customer geolocation data, which will be merged later.

In [14]:
# seller geo
geo_seller = geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'geolocation_lat': 'seller_lat',
    'geolocation_lng': 'seller_lng',
    'geolocation_city': 'seller_geo_city',
    'geolocation_state': 'seller_geo_state',
})
df = df.merge(geo_seller, on='seller_zip_code_prefix', how='left')
print('+ geolocation (seller): ', df.shape)

+ geolocation (seller):  (113425, 43)


Similarly to `geo_seller`, the `geo_customer` dataset was created by renaming its geolocation columns to prevent naming conflicts with the seller geolocation data that was merged previously.

In [15]:
# customer geo
geo_customer = geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'geolocation_lat': 'customer_lat',
    'geolocation_lng': 'customer_lng',
    'geolocation_city': 'customer_geo_city',
    'geolocation_state': 'customer_geo_state',
})
df = df.merge(geo_customer, on='customer_zip_code_prefix', how='left')
print('+ geolocation (customer): ', df.shape)

display(df.head())

+ geolocation (customer):  (113425, 47)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,credit_card,2.0,20.59,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,-23.680729,-46.444238,maua,SP,-23.576983,-46.587161,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,0.0,0.00,4.0,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,-19.807681,-43.980427,belo horizonte,MG,-12.177924,-44.660711,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,0.0,0.00,5.0,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,-21.363502,-48.229601,guariba,SP,-16.745150,-48.514783,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,credit_card,0.0,0.00,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,-19.837682,-43.924053,belo horizonte,MG,-5.774190,-35.271143,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,credit_card,0.0,0.00,5.0,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51,-23.543395,-46.262086,mogi das cruzes,SP,-23.676370,-46.514627,santo andre,SP


## **2.7 Exporting Merged Dataset**

In [16]:
df.to_csv('merged.csv', index=False)
print('Saved with shape', df.shape)

Saved with shape (113425, 47)


# **Section 3: Data Cleaning**